In [ ]:
import requests
import sqlite3
import pandas as pd

# 1. Data Acquisition: Fetch trending cryptocurrencies from CoinGecko
url = 'https://api.coingecko.com/api/v3/search/trending'
headers = {'accept': 'application/json'}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    coin_data = []
    for coin in data.get('coins', []):
        coin_item = coin.get('item', {})
        coin_data.append({
            'id': coin_item.get('id', ''),
            'name': coin_item.get('name', ''),
            'symbol': coin_item.get('symbol', ''),
            'market_cap_rank': coin_item.get('market_cap_rank', 0),
            'score': coin_item.get('score', 0)
        })
    # Convert to DataFrame for easy viewing/manipulation
    df = pd.DataFrame(coin_data)
    print(df)
else:
    print(f"API request failed with status code: {response.status_code}")
    coin_data = []

# 2. Database Setup: Create SQLite database and table
conn = sqlite3.connect('crypto_trending.db')
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS trending_coins (
        id TEXT PRIMARY KEY,
        name TEXT,
        symbol TEXT,
        market_cap_rank INTEGER,
        score INTEGER
    )
''')
conn.commit()

# 3. Data Storage: Insert API data into database
for coin in coin_data:
    cursor.execute('''
        INSERT OR REPLACE INTO trending_coins (id, name, symbol, market_cap_rank, score)
        VALUES (?, ?, ?, ?, ?)
    ''', (coin['id'], coin['name'], coin['symbol'], coin['market_cap_rank'], coin['score']))
conn.commit()

# 4. Data Retrieval: Fetch stored data for verification
cursor.execute('SELECT * FROM trending_coins ORDER BY market_cap_rank')
rows = cursor.fetchall()
for row in rows:
    print(row)

# Optional: Convert SQL results to DataFrame for analytics
trending_df = pd.read_sql('SELECT * FROM trending_coins', conn)
print(trending_df)

# Clean up
conn.close()


                   id               name symbol  market_cap_rank  score
0               zcash              Zcash    ZEC               32      0
1           bittensor          Bittensor    TAO               41      1
2       chainopera-ai      ChainOpera AI   COAI              238      2
3               ai16z              ai16z  AI16Z              563      3
4             aster-2              Aster  ASTER               67      4
5      pudgy-penguins     Pudgy Penguins  PENGU              103      5
6             bitcoin            Bitcoin    BTC                1      6
7              solana             Solana    SOL                6      7
8            ethereum           Ethereum    ETH                2      8
9              plasma             Plasma    XPL              155      9
10  aerodrome-finance  Aerodrome Finance   AERO              112     10
11  velodrome-finance  Velodrome Finance   VELO              795     11
12           pump-fun           Pump.fun   PUMP               79